# 💼 Mini Project 1 (LangChain): HireMatch — Dynamic Talent Screening Agent

Welcome to **Mini Project 1 (HireMatch)**. This project dynamically processes candidate resumes (from file paths or URL links) without hardcoding any candidate data, across all 5 core LangChain modules:

1. **Module 1**: Environment & Agent Setup (`create_agent`)
2. **Module 2**: Multi-Provider Model Initialization (`init_chat_model`) & Custom Tools (`@tool`)
3. **Module 3**: Messages & State (`SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`)
4. **Module 4**: Dynamic Structured Output Extraction with Pydantic (`with_structured_output`)
5. **Module 5**: Agent Governance with Middleware (`SummarizationMiddleware` & `HumanInTheLoopMiddleware`)

## 1️⃣ Environment & Setup

In [ ]:
import os
import sys
import time
from typing import Literal, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY', '')
MODEL_NAME = 'groq:openai/gpt-oss-120b'
print("Environment initialized successfully!")

## 2️⃣ Dynamic Custom Tools (`@tool`)

In [ ]:
@tool
def read_submitted_resume(file_path_or_url: str) -> str:
    """Read and parse candidate resume text from a submitted file path or URL link."""
    if os.path.exists(file_path_or_url):
        with open(file_path_or_url, 'r', encoding='utf-8') as f:
            return f"--- RESUME CONTENT ({file_path_or_url}) ---\n{f.read()}"
    return f"Resume input received: {file_path_or_url}"

@tool
def check_interviewer_availability(interviewer_name: str, date: str) -> str:
    """Check calendar availability of a hiring manager for a specific date."""
    return f"Availability for {interviewer_name} on {date}: Available at 10:00 AM EST and 2:00 PM EST."

@tool
def send_interview_invite(candidate_email: str, interviewer_name: str, slot: str) -> str:
    """Send official interview invitation email to candidate."""
    return f"SUCCESS: Interview invitation sent to {candidate_email} with {interviewer_name} for {slot}."

@tool
def send_rejection_notice(candidate_email: str, reason: str) -> str:
    """Send polite rejection notice to candidate."""
    return f"SUCCESS: Rejection notice sent to {candidate_email}. Reason logged: {reason}"

print("Dynamic tools registered successfully!")

## 3️⃣ Dynamic Pydantic Schema for Candidate Evaluation

In [ ]:
class CandidateEvaluation(BaseModel):
    candidate_name: str = Field(description="Full name of the candidate extracted from resume")
    candidate_email: str = Field(description="Email address of the candidate extracted from resume")
    skill_match_score: int = Field(description="Technical skill match score out of 100 based on experience")
    experience_level: Literal["Junior", "Mid-Level", "Senior", "Lead"] = Field(description="Assessed experience level")
    expected_salary: str = Field(description="Expected salary mentioned in resume or market assessment")
    recommendation: Literal["Hire", "Interview", "Reject", "Hold"] = Field(description="Recruitment decision recommendation")
    key_strengths: List[str] = Field(description="List of key technical strengths extracted from resume")
    summary: str = Field(description="Executive evaluation summary of candidate resume")

print("CandidateEvaluation schema compiled!")

## 4️⃣ Agent Construction with Middleware & Governance

In [ ]:
agent = create_agent(
    model=MODEL_NAME,
    tools=[read_submitted_resume, check_interviewer_availability, send_interview_invite, send_rejection_notice],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=MODEL_NAME,
            trigger=("tokens", 600),
            keep=("tokens", 250)
        ),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_interview_invite": {"allowed_decisions": ["approve", "edit", "reject"]},
                "send_rejection_notice": {"allowed_decisions": ["approve", "edit", "reject"]},
                "read_submitted_resume": False,
                "check_interviewer_availability": False,
            }
        )
    ]
)
print("HireMatch Agent compiled with Summarization & Human-In-The-Loop Middleware!")

## 5️⃣ Dynamic Processing Function (File / URL Resume Input)

In [ ]:
def process_resume(file_path: str, interviewer: str = "Sarah Connor", date: str = "2026-09-01"):
    with open(file_path, "r", encoding="utf-8") as f:
        resume_text = f.read()
    
    print(f"\n📂 Processing Resume: {file_path}")
    model = init_chat_model(MODEL_NAME)
    structured_llm = model.with_structured_output(CandidateEvaluation)
    
    eval_card = structured_llm.invoke(f"Parse and evaluate resume text:\n{resume_text}")
    print(f"✨ Extracted Card: {eval_card.candidate_name} ({eval_card.candidate_email}) - Score: {eval_card.skill_match_score}/100 - Rec: {eval_card.recommendation}")
    
    thread_id = f"session_{eval_card.candidate_name.lower().replace(' ', '_')}"
    config = {"configurable": {"thread_id": thread_id}}
    
    system_prompt = "You are HireMatch AI. Read resumes, check availability, and invite qualified candidates."
    user_req = f"Read resume '{file_path}'. Candidate is '{eval_card.candidate_name}' ({eval_card.candidate_email}). Recommendation is '{eval_card.recommendation}'. If qualified, check '{interviewer}' on '{date}' and invite for 10:00 AM EST."
    
    result = agent.invoke({"messages": [SystemMessage(content=system_prompt), HumanMessage(content=user_req)]}, config=config)
    
    if "__interrupt__" in result:
        print("⚠️ [HITL INTERRUPT DETECTED] Approving invitation...")
        final_res = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
        print("✅ Response:", final_res["messages"][-1].content)
    else:
        print("✅ Response:", result["messages"][-1].content)

# Test on Alex Rivera's resume file
process_resume('../resumes/alex_rivera_resume.txt')